# Notebook 2 — Target Construction

**Goal**: Define the binary default target variable for Estonian loans.


**Business logic**:
 - A loan is "defaulted within 12 months" if lender flagged it (DefaultDate exists)
     <br>AND the gap between LoanDate and DefaultDate is ≤ 365 days.
     
- Only loans issued before the cutoff date (snapshot − 365d) enter the modelling
   <br>universe, because newer loans haven't had a full year to potentially default.

# Load Libraries

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import sys, os

# Add project src/ to path
sys.path.insert(0, os.path.abspath('../src'))

from utils.config import PATHS, SNAPSHOT_DATE, HORIZON_DAYS, TARGET_COL
from utils.target import (
    build_modelling_universe,
    create_default_target,
    validate_target,
)


# 1. Load Data

In [2]:
df_ee = pd.read_parquet(PATHS['cleaned'])
print(f"Loaded: {len(df_ee):,} rows × {df_ee.shape[1]} columns")
print(f"LoanDate range: {df_ee['LoanDate'].min()} → {df_ee['LoanDate'].max()}")

Loaded: 162,573 rows × 43 columns
LoanDate range: 2009-02-28 00:00:00 → 2023-10-14 00:00:00


# 2. Build Modelling Universe
We restrict to loans issued at least 365 days before the snapshot date.

<br>This ensures every loan has had a full 12-month observation window.

<br>Without this, recent loans would be labelled as "non-default" simply

<br>because they haven't had time to default yet

In [3]:
df_model, cutoff_date = build_modelling_universe(
    df_ee,
    snapshot_date=SNAPSHOT_DATE,
    horizon_days=HORIZON_DAYS,
)

Modelling universe: 145,029 loans
  LoanDate range: 2009-02-28 → 2022-10-15
  Cutoff date: 2022-10-15 (snapshot 2023-10-15 - 365d)


# 3. Create Target Variable

In [4]:
df_model = create_default_target(
    df_model,
    horizon_days=HORIZON_DAYS,
    target_col=TARGET_COL,
)

Target 'into_12m_default_ind' created:
  Default (1): 22,248 (15.34%)
  Non-default (0): 122,781 (84.66%)


As expected the credit risk dataset is imbalanced with non-defaults taking up most of the proportion

# 4. Validate Target

Three sanity checks to catch  bugs:
   1. No loan can have into_12m_default_ind=1 if DefaultDate is null

   2. All into_12m_default_ind=1 loans must have days_to_default in [0, 365]

   3. No DefaultDate should be before LoanDate (negative gap)

In [5]:
is_valid = validate_target(
    df_model,
    horizon_days=HORIZON_DAYS,
    target_col=TARGET_COL,
)

[1] No null DefaultDate with target=1 : PASS
[2] All target=1 within [0, 365] days : PASS
[3] No DefaultDate before LoanDate    : PASS

✓ All target validation checks passed


# 5. Save Dataset

In [6]:
df_model.to_parquet(PATHS['target_built'], index=False)
print(f"\n✓ Saved → {PATHS['target_built']}")
print(f"  Shape: {df_model.shape[0]:,} rows × {df_model.shape[1]} columns")
print(f"  Target: '{TARGET_COL}'")


✓ Saved → /home/me/Documents/Coding/Projects/pd_model/data/02_processed/02_estonia_target_creation.parquet
  Shape: 145,029 rows × 45 columns
  Target: 'into_12m_default_ind'
